In [ ]:
import pandas as pd
import numpy as np
import requests
import json

In [2]:
backups = pd.read_csv("https://raw.githubusercontent.com/datarescueproject/portal/refs/heads/main/baserow_exports/datarescue_backups.csv")
datasets = pd.read_csv("https://raw.githubusercontent.com/datarescueproject/portal/refs/heads/main/baserow_exports/datarescue_datasets.csv")

In [3]:
datasets.columns = datasets.columns.str.lower()
datasets = datasets.fillna('')
datasets.head()

,dataset,notes,dataset_id,url,websites,organization,agency,last_modified
0,Billion-Dollar Weather and Climate Disasters,,1,https://www.ncei.noaa.gov/access/billions/mapping,ncei.noaa.gov,National Oceanic and Atmospheric Administration,Department of Commerce,2025-02-10
1,American Communities Survey (ACS),,3,https://www.census.gov/programs-surveys/acs,census.gov,Census Bureau,Department of Commerce,2025-03-03
2,BLS Downloads,,6,https://download.bls.gov,download.bls.gov,Bureau of Labor Statistics,Department of Labor,2025-02-10
3,CDC FTP,,7,https://ftp.cdc.gov/,ftp.cdc.gov,Centers for Disease Control and Prevention,Department of Health and Human Services,2025-02-10
4,US Census Bureau FTP,,8,ftp://ftp.census.gov,census.gov,Census Bureau,Department of Commerce,2025-02-11


In [4]:
agencies = datasets.agency.value_counts().keys()
categories = categories = ['arts-culture-history','health-human-services',
'budget-finance','parks-recreation','economy','planning-zoning',
'education','public-safety','elections-politics','real-estate-land-records',
'environment','transportation','food','uncategorized']


In [5]:
agency_to_category = {
    'Department of Health and Human Services': 'Health / Human Services',
    'Department of Commerce': 'Economy',
    'Department of Housing and Urban Development': 'Real Estate / Land Records',
    'Department of Veterans Affairs': 'Health / Human Services',
    'National Endowment for the Humanities': 'Arts / Culture / History',
    'AmeriCorps': 'Public Safety',
    'Department of Education': 'Education',
    'Federal Mediation and Conciliation Service': 'Economy',
    'Department of Homeland Security': 'Public Safety',
    'Department of Energy': 'Environment',
    'National Labor Relations Board': 'Economy',
    'Environmental Protection Agency': 'Environment',
    'Consumer Financial Protection Bureau': 'Budget / Finance',
    'Federal Housing Finance Agency': 'Real Estate / Land Records',
    'Department of the Treasury': 'Budget / Finance',
    'Institute of Museum and Library Services': 'Arts / Culture / History',
    'Department of the Interior': 'Parks / Recreation',
    'General Services Administration': 'Economy',
    'Department of Labor': 'Economy',
    'U.S. Agency for International Development': 'Health / Human Services',
    'Department of Transportation': 'Transportation',
    'National Aeronautics and Space Administration': 'Environment',
    '': 'Uncategorized',
    'Department of Justice': 'Public Safety',
    'Department of the Interior, National Parks Service': 'Parks / Recreation',
    'Department of State': 'Elections / Politics',
    'National Science Foundation': 'Education',
    'Department of Health and Human Services, Department of Commerce': 'Health / Human Services',
    'Consumer Financial Protection Bureau, Federal Housing Finance Agency': 'Budget / Finance',
    'U.S. Department of Agriculture': 'Food',
    'Office of Management and Budget': 'Budget / Finance'
}

In [6]:
backups.columns = backups.columns.str.lower()
backups = backups.fillna('')
backups.head()

,dataset,dataset_id,status,url,source_website,organization,agency,download_date,size,maintainer,download_location,file_type,notes,metadata_available,metadata_url
0,Billion-Dollar Weather and Climate Disasters,1,Finished,https://www.ncei.noaa.gov/access/billions/mapping,ncei.noaa.gov,National Oceanic and Atmospheric Administration,Department of Commerce,2025-02-10,0.15,HD,https://dataverse.harvard.edu/dataset.xhtml?pe...,ZIP,,yes,https://dataverse.harvard.edu/dataset.xhtml?pe...
1,BLS Downloads,6,Finished,https://download.bls.gov,download.bls.gov,Bureau of Labor Statistics,Department of Labor,2025-02-01,47.0,DRP,,,,,
2,CDC FTP,7,Finished,https://ftp.cdc.gov/,ftp.cdc.gov,Centers for Disease Control and Prevention,Department of Health and Human Services,2025-02-01,213.0,DRP,,,,,
3,US Census Bureau FTP,8,Finished,ftp://ftp.census.gov,census.gov,Census Bureau,Department of Commerce,2025-02-01,180.0,DRP,,,"Partial download, server is back online but co...",,
4,National Hurricane Center (NHC),9,Finished,https://www.nhc.noaa.gov/archive,nhc.noaa.gov,NOAA/National Hurricane Center,Department of Commerce,2025-02-06,61.0,DRP,,,,,


In [11]:
import re
def slugify(string):
    string = clean_text(string)
    # Remove special characters
    string = re.sub(r'[^\w\s-]', '', string)
    # Replace spaces with hyphens
    string = re.sub(r'\s+', '-', string)
    # Convert to lowercase
    string = string.lower()
    return string

def clean_text(string):
    # Remove URL prefixes like http:// or https://
    # string = re.sub(r'http[s]?://', '', string)
    # Remove escape strings like \n
    string = string.replace('\n', '').replace('\r', '').replace('\t', '')
    # Remove leading '-'
    string = re.sub(r'^-', '', string)
    # Replace ':' with '-'
    string = string.replace(':', '')
    return string

def get_dataset_categories(agency):
    return agency_to_category[agency]

In [8]:
def get_metadata_availability(dataset_id):
    """
    This function checks the metadata availability for dataset_id 432 in the backups dataframe.
    It returns "Yes" if metadata is available, "Under Review" if it needs review, and "No" otherwise.
    """
    md_avl = backups[backups.dataset_id == dataset_id].metadata_available.values
    if "yes" in md_avl:

        return "Yes",backups[backups.dataset_id == dataset_id].metadata_url.values[0]
    elif "needs review" in md_avl:
        return "Under Review",""
    else:
        return "No",""

def create_dataset_md(row):
    if row['organization'] == '':
      row['organization'] = 'Unknown'
    ## Defining the schema, filename and path
    schema = 'data_rescue_project'
    dataset_filename = slugify(row['dataset'])
    dataset_path = "_datasets"
    org_filename = slugify(row['organization'])
    org_path = "_organizations"

    ## Get backups for each dataset
    data_backups = backups[backups.dataset == row['dataset']]
    metadata_available, metadata_url = get_metadata_availability(row['dataset_id'])
    ## Creating the dataset markdown file
    ## Dataset-level information
    dataset_md = "---\n"
    dataset_md += f"schema: {schema} \n"
    dataset_md += f"title: {clean_text(row['dataset'])}\n"
    dataset_md += f"organization: {clean_text(row['organization'])}\n"
    dataset_md += f"agency: {clean_text(row['agency'])}\n"
    dataset_md += f"websites: {row['websites']}\n"
    dataset_md += f"data_source: {row['url']}\n"
    dataset_md += f"description: {clean_text(row['notes'])}\n"
    dataset_md += f"last_modified: {row['last_modified']}\n"
    ## Check if any backups have metadata available and populate
    dataset_md += f"metadata_available: {metadata_available}\n"
    dataset_md += f"metadata_url: {metadata_url}\n"
    dataset_md += f"category:\n"
    dataset_md += f"  - {get_dataset_category(clean_text(row['agency']))}\n"

    dataset_md += f"resources:\n"
    ## Resource-level information
    for index, backup_row in data_backups.iterrows():
      dataset_md += f"  - id: {index}\n"
      dataset_md += f"    url: {backup_row['download_location']}\n"
      dataset_md += f"    format: {clean_text(backup_row['file_type'])}\n"
      dataset_md += f"    status: {clean_text(backup_row['status'])}\n"
      dataset_md += f"    size: {backup_row['size']}\n"
      dataset_md += f"    download_date: {backup_row['download_date']}\n"
      dataset_md += f"    maintainer: {clean_text(backup_row['maintainer'])}\n"
      dataset_md += f"    notes: {clean_text(backup_row['notes'])}\n"
    dataset_md += "---\n"
      
    ## Writing the dataset markdown file
    with open(f'{dataset_path}/{dataset_filename}.md', 'w') as output:
      output.write(dataset_md)
    
    ## Creating the organization markdown file
    org_md = "---\n"
    org_md += f"title: {clean_text(row['organization'])} \n" 
    org_md += f"description: \n" 
    org_md += "---\n"

    ## Writing the organization markdown file
    with open(f'{org_path}/{org_filename}.md', 'w') as output:
      output.write(org_md)

In [9]:
datasets.shape

(731, 8)

In [13]:
datasets.apply(create_dataset_md, axis=1)

0      None
1      None
2      None
3      None
4      None
       ... 
726    None
727    None
728    None
729    None
730    None
Length: 731, dtype: object

In [10]:
import os
def remove_files_os(dir_path):
    for filename in os.listdir(dir_path):
        file_path = os.path.join(dir_path, filename)
        if os.path.isfile(file_path):
            os.remove(file_path)


In [5]:
a = []
if not a:
    print("test")

test


In [ ]:
def replace_multiple_spaces(string):
    return re.sub(r'\s+', ' ', string)

# Example usage
example_string = "This   is  a   string    with multiple   spaces."
cleaned_string = replace_multiple_spaces(example_string)
print(cleaned_string)

In [11]:
remove_files_os('_datasets')

In [8]:
def clean_text(string):
    # Remove URL prefixes like http:// or https://
    # string = re.sub(r'http[s]?://', '', string)
    # Remove escape strings like \n
    string = string.replace('\n', '').replace('\r', '').replace('\t', '')
    # Remove multiple spaces
    string = re.sub(r'\s+', ' ', string)
    # Remove leading spl. characters
    string = re.sub(r'^[^a-zA-Z]+', '', string)
    # string = string.lstrip(',')
    string = re.sub(r'^-', '', string)
    # Remove leading and trailing ':'
    string = string.rstrip(':')
    string = re.sub(r'(?<!http)(?<!https):', '', string)
    
    return string

In [9]:
test_string = ",https://www.ncei.noaa.gov/metadata/geoportal/rest/metadata/item/gov.noaa.ncdcC01557/html#"
clean_text(test_string)

'https://www.ncei.noaa.gov/metadata/geoportal/rest/metadata/item/gov.noaa.ncdcC01557/html#'

In [ ]:
test_string = test_string.lstrip(',')
print(test_string)

In [3]:
%load_ext autoreload
%autoreload 2

from create_markdowns import *
import os

In [8]:
def remove_files_os(dir_path):
    for filename in os.listdir(dir_path):
        file_path = os.path.join(dir_path, filename)
        if os.path.isfile(file_path):
            os.remove(file_path)
            
# Remove files in _datasets and _organizations
remove_files_os('../_datasets')
remove_files_os('../_organizations')
remove_files_os('../_dataset_categories')

create_markdowns()

In [5]:
organizations = pd.read_csv("https://raw.githubusercontent.com/datarescueproject/portal/refs/heads/main/baserow_exports/datarescue_organizations.csv")
print(organizations[organizations['Organizations'] == 'National Oceanic and Atmospheric Administration']['Categories'].str.split(';').values[0])
print(organizations[organizations['Organizations'] == 'Department of the Interior']['Categories'].str.split(';').values[0])

['Climate & Environment']
['Climate & Environment', 'Humanitarian & Disaster Relief']


In [6]:
organizations

,Organizations,Categories
0,American Battle Monuments Commission,Arts & Culture
1,Barry Goldwater Scholarship and Excellence in ...,Education
2,Consumer Financial Protection Bureau,Business & Economy
3,Delta Regional Authority,Business & Economy
4,Denali Commission,Business & Economy;Infrastructure
...,...,...
416,Federal Mediation and Conciliation Service,Labor & Employment
417,Health Resources and Services Administration,Health & Healthcare;Social Services
418,Institute of International Education,Education
419,U.S. Patent and Trademark Office,Business & Economy


In [ ]:
combined_array = organizations.values.flatten()
print(combined_array)

In [6]:
backups = pd.read_csv("https://raw.githubusercontent.com/datarescueproject/portal/refs/heads/main/baserow_exports/datarescue_backups.csv")
datasets = pd.read_csv("https://raw.githubusercontent.com/datarescueproject/portal/refs/heads/main/baserow_exports/datarescue_datasets.csv")
organizations = pd.read_csv("https://raw.githubusercontent.com/datarescueproject/portal/refs/heads/main/baserow_exports/datarescue_organizations.csv")

backups.columns = backups.columns.str.lower()
backups = backups.fillna('')
backups.head()

datasets.columns = datasets.columns.str.lower()
datasets = datasets.fillna('')
datasets.head()

organizations = organizations.fillna('')

In [72]:
datasets = datasets[datasets['dataset'].str.contains("Environmental Justice")]

In [76]:
create_dataset_md(datasets.loc[54],backups, organizations)

In [82]:
row = datasets.loc[54]

In [83]:
if row['organization'] == '':
    row['organization'] = 'Unknown'
# Defining the schema, filename and path
schema = 'data_rescue_project'
dataset_filename = slugify(row['dataset'])
dataset_path = "../_datasets"
org_filename = slugify(row['organization'])
org_path = "../_organizations"

# Get backups for each dataset
data_backups = backups[backups.dataset == row['dataset']]
metadata_available, metadata_url = get_metadata_availability(row['dataset_id'], data_backups)
# Creating the dataset markdown file
# Dataset-level information
dataset_md = "---\n"
dataset_md += f"schema: {schema} \n"
dataset_md += f"title: {clean_text(row['dataset'])}\n"
dataset_md += f"organization: {clean_text(row['organization'])}\n"
dataset_md += f"agency: {clean_text(row['agency'])}\n"
dataset_md += f"websites: {clean_text(row['websites'])}\n"
dataset_md += f"data_source: {clean_text(row['url'])}\n"
dataset_md += f"description: {clean_text(row['notes'])}\n"
dataset_md += f"last_modified: {row['last_modified']}\n"
# Check if any backups have metadata available and populate
dataset_md += f"metadata_available: {metadata_available}\n"
dataset_md += f"metadata_url: {clean_text(metadata_url)}\n"
dataset_md += "category:\n"
cats = get_dataset_category(row, organizations)

for cat in cats:
    dataset_md += f"  - {cat} \n"
    
dataset_md += "resources:\n"
# Resource-level information
for index, backup_row in data_backups.iterrows():
    dataset_md += f"  - id: {index}\n"
    dataset_md += f"    url: {clean_text(backup_row['download_location'])}\n"
    dataset_md += f"    format: {clean_text(backup_row['file_type'])}\n"
    dataset_md += f"    status: {clean_text(backup_row['status'])}\n"
    dataset_md += f"    size: {backup_row['size']}\n"
    dataset_md += f"    download_date: {backup_row['download_date']}\n"
    dataset_md += f"    maintainer: {clean_text(backup_row['maintainer'])}\n"
    dataset_md += f"    notes: {clean_text(backup_row['notes'])}\n"
dataset_md += "---\n"
    
# Writing the dataset markdown file
with open(f'{dataset_path}/{dataset_filename}.md', 'w') as output:
    output.write(dataset_md)

# Creating the organization markdown file
org_md = "---\n"
org_md += f"title: {clean_text(row['organization'])} \n" 
org_md += "description: \n" 
org_md += "---\n"

# Writing the organization markdown file
with open(f'{org_path}/{org_filename}.md', 'w') as output:
    output.write(org_md)

In [79]:
def get_dataset_category(row, organizations):
    # Check if dataset has category override
    categories = eval(row['categories'])
    if categories:
        cats = [a['value'] for a in categories]
    # Check if we don't have organization info
    elif row['organization'] == 'Unknown':
        cats = ['Uncategorized']
    else:
        # Get categories from organization
        cats_from_org = organizations[organizations['Organizations'] == row['organization']]['Categories'].values
        cats = []
        [cats.extend(v.split(';')) for v in cats_from_org]      
        cats = list(set(cats))
        if cats == ['']:
            cats = ['Uncategorized']
        else:
            cats = [cat for cat in cats if cat != '']
    
    return cats

In [80]:
get_dataset_category(datasets.loc[54], organizations)

['Climate & Environment', 'Health & Healthcare']

In [59]:
datasets = datasets[datasets['dataset'].str.startswith('20')]

In [60]:
datasets.apply(create_dataset_md, axis=1, args=(backups, organizations))

393    None
395    None
396    None
397    None
398    None
399    None
400    None
401    None
403    None
404    None
405    None
407    None
408    None
409    None
410    None
416    None
640    None
662    None
663    None
665    None
666    None
667    None
696    None
697    None
698    None
798    None
799    None
dtype: object

In [ ]:
def get_dataset_category(row):
    # Check if dataset has category override
    categories = eval(row['categories'])
    if categories:
        cats = [a['value'] for a in categories]
    # Check if we don't have organization info
    elif row['organization'] == 'Unknown':
        cats = ['Uncategorized']
    else:
        # Get categories from organization
        cats_from_org = organizations[organizations['Organizations'] == row['organization']]['Categories'].values
        cats = []
        [cats.extend(v.split(';')) for v in cats_from_org]      
        cats = list(set(cats))
        if cats == ['']:
            cats = ['Uncategorized']
        else:
            cats = [cat for cat in cats if cat != '']
        

[None, None]

In [51]:
a = ''
len(a.split(';'))

1

In [9]:
import numpy as np
organizations[organizations['Categories']== np.nan]

,Organizations,Categories


In [ ]:
BASEROW_ACCESS_TOKEN = 'your_baserow_access_token_here'

def stringify_arr_vals(arr):
    return ';'.join([i['value'] for i in arr])

def get_results_json(url):
    table = requests.get(
        url,
        headers={
            "Authorization": f"Token {BASEROW_ACCESS_TOKEN}"
        }
    )

    res = table.json()['results']
    if table.json()['next'] is not None:
        res.extend(get_results_json(table.json()['next']))

    return res

# categories = pd.DataFrame(get_results_json("https://baserow.datarescueproject.org/api/database/rows/table/732/?user_field_names=true"))[['Name', 'Active']]
# organizations = pd.DataFrame(get_results_json("https://baserow.datarescueproject.org/api/database/rows/table/638/?user_field_names=true"))[['Organizations', 'Categories']]
# organizations['Categories'] = organizations['Categories'].apply(lambda x: stringify_arr_vals(x))
# categories.to_csv("baserow_exports/datarescue_categories.csv", index=False)
# organizations.to_csv("baserow_exports/datarescue_organizations.csv", index=False)

In [8]:
# agencies = pd.DataFrame(get_results_json("https://baserow.datarescueproject.org/api/database/rows/table/645/?user_field_names=true"))[['Name']]
agencies.to_csv("../baserow_exports/datarescue_agencies.csv", index=False)

In [13]:
def create_agency_md(row):
    """
    This function creates a markdown file for each agency.
    """
    agency_path = "../_agencies"
    agency_filename = slugify(row['Name'])

    # Creating the agency markdown file
    agency_md = "---\n"
    agency_md += f"title: {clean_text(row['Name'])} \n"
    agency_md += "description: \n"
    agency_md += "---\n"

    # Writing the agency markdown file
    with open(f'{agency_path}/{agency_filename}.md', 'w') as output:
        output.write(agency_md)
      

In [14]:
agencies.apply(create_agency_md, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
72    None
73    None
74    None
75    None
76    None
Length: 77, dtype: object

In [23]:
datasets = pd.read_csv("https://raw.githubusercontent.com/datarescueproject/portal/refs/heads/main/baserow_exports/datarescue_datasets.csv")

In [24]:
datasets

,dataset,notes,dataset_id,url,websites,organization,agency,categories,last_modified
0,Billion-Dollar Weather and Climate Disasters,NaN,1,https://www.ncei.noaa.gov/access/billions/mapping,ncei.noaa.gov,National Oceanic and Atmospheric Administration,Department of Commerce,[],2025-02-10
1,American Communities Survey (ACS),NaN,3,https://www.census.gov/programs-surveys/acs,census.gov,Census Bureau,Department of Commerce,[],2025-03-03
2,Enforcement and Compliance History Online (ECHO),NaN,13,https://echo.epa.gov/files/echodownloads,echo.epa.gov,Environmental Protection Agency,Environmental Protection Agency,[],2025-03-26
3,College Scorecard,NaN,15,https://collegescorecard.ed.gov/data/,collegescorecard.ed.gov,Office of Chief Information Officer,Department of Education,[],2025-02-11
4,Campus Safety and Security,NaN,16,https://ope.ed.gov/campussafety/#/datafile/list,ope.ed.gov,Office of Chief Information Officer,Department of Education,[],2025-02-11
...,...,...,...,...,...,...,...,...,...
2929,National Accounts (NIPA) Archive,NaN,5057,https://apps.bea.gov/histdata/histChildLevels....,bea.gov,Bureau of Economic Analysis,Department of Commerce,[],2026-02-22
2930,Fixed Asset Archive,NaN,5058,https://apps.bea.gov/histdata/histChildLevels....,bea.gov,Bureau of Economic Analysis,Department of Commerce,[],2026-02-22
2931,"National Center for HIV, Viral Hepatitis, STD,...",NaN,5059,https://gis.cdc.gov/grasp/nchhstpatlas/tables....,gis.cdc.gov,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,[],2026-02-26
2932,Fedscope Datasets from OPM,Diversity data removed from FedScope Data Cube...,5060,https://www.fedscope.opm.gov/,fedscope.opm.gov,Office of Personnel Management,Office of Personnel Management,[],2026-03-23


In [10]:
datasets = pd.read_csv("https://raw.githubusercontent.com/datarescueproject/portal/refs/heads/main/baserow_exports/datarescue_datasets.csv")
datasets.head()

,dataset,notes,dataset_id,url,websites,organization,agency,categories,last_modified,dataset_source_status
0,Billion-Dollar Weather and Climate Disasters,NaN,1,https://www.ncei.noaa.gov/access/billions/mapping,ncei.noaa.gov,National Oceanic and Atmospheric Administration,Department of Commerce,[],2025-02-10,NaN
1,American Communities Survey (ACS),NaN,3,https://www.census.gov/programs-surveys/acs,census.gov,Census Bureau,Department of Commerce,[],2025-03-03,NaN
2,Enforcement and Compliance History Online (ECHO),NaN,13,https://echo.epa.gov/files/echodownloads,echo.epa.gov,Environmental Protection Agency,Environmental Protection Agency,[],2025-03-26,NaN
3,College Scorecard,NaN,15,https://collegescorecard.ed.gov/data/,collegescorecard.ed.gov,Office of Chief Information Officer,Department of Education,[],2025-02-11,NaN
4,Campus Safety and Security,NaN,16,https://ope.ed.gov/campussafety/#/datafile/list,ope.ed.gov,Office of Chief Information Officer,Department of Education,[],2025-02-11,NaN


In [11]:
datasets.shape

(2934, 10)

In [12]:
import warnings
warnings.filterwarnings('ignore', category=requests.packages.urllib3.exceptions.InsecureRequestWarning)

In [14]:
ds_urls = datasets['url'].tolist()
url_statuses = []
url_status_notes = []
error_code_mapping = {    
    401: "Unauthorized: Authentication is required and has failed or has not yet been provided.",
    403: "Forbidden: Might be forbidden to access programmatically, but can be accessed via a browser.",
    404: "Not Found: The server can not find the requested resource.",
    500: "Internal Server Error: The server has encountered a situation it doesn't know how to handle.",
    502: "Bad Gateway: The server was acting as a gateway or proxy and received an invalid response from the upstream server.",
    503: "Service Unavailable: Unable to verify programatically, but might able to be accessed via a browser.",
}
for url in ds_urls:
    try:
        response = requests.head(url, allow_redirects=True, timeout=10, verify=False)
        if response.status_code == 200:
            # print(f"URL is valid: {url}")
            url_statuses.append(response.status_code)
            url_status_notes.append("Valid URL")
        else:
            print(f"URL returned status code {response.status_code}: {url}")
            url_statuses.append(response.status_code)
            url_status_notes.append(error_code_mapping[response.status_code] if response.status_code in error_code_mapping else "Other Error")
    except requests.exceptions.RequestException as e:
        print(f"Error accessing URL {url}: {e}")
        url_statuses.append(0)
        url_status_notes.append(f'error ({e})')

URL returned status code 404: https://www.fema.gov/about/openfema/data-sets/national-household-survey
URL returned status code 404: https://www.fema.gov/about/openfema/data-sets/grant-programs-directorate-preparedness-non-disasterassistance-firefighter-grants
URL returned status code 403: https://www.census.gov/naics/?input=1531&year=2022
Error accessing URL https://justice40tool.lbl.gov/: HTTPSConnectionPool(host='justice40tool.lbl.gov', port=443): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x1628756d0>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known'))
URL returned status code 403: https://www.phmsa.dot.gov/data-and-statistics/pipeline/gas-distribution-gas-gathering-gas-transmission-hazardous-liquids
URL returned status code 503: https://www.eia.gov/outlooks/aeo/data/browser/
URL returned status code 503: https://www.eia.gov/consumption/commercial/data/2018/
URL returned s

In [15]:
datasets['url_status'] = url_statuses
datasets['url_status_notes'] = url_status_notes
datasets.to_csv("../baserow_exports/dataset_urlcheck_status.csv", index=False)

In [1]:
import json
import pandas as pd
import requests

In [ ]:
try:
    resp = requests.get(url="https://findgovdata.org/files/last_seen.json")
    

{'generated': '2026-07-09T08:17:04.597961Z',
 'filter': 'last_seen < "2026-07-09"',
 'documents_count': 2279,
 'documents': [{'id': 'ba1f2b82-6222-5a74-9255-862e2ca6fac7',
   'timestamp': '2022-02-22T00:00:00Z',
   'tags': None,
   'name': 'Commercial Harvest Sites',
   'description': 'Locations of commercial shellfish harvest sites (shellfish harvesters, shellstock shippers, and shucker packers) in Washington State. Also displayed on the Commercial Shellfish Map Viewer here: https://fortress.wa.gov/doh/oswpviewer/index.html Relay and wet storage sites are not included.',
   'content': None,
   'metadata': {'organization': {'organization_type': 'State Government',
     'logo': 'https://raw.githubusercontent.com/GSA/logo/refs/heads/master/wa-opendata.png',
     'name': 'State of Washington',
     'slug': 'washington',
     'description': None,
     'aliases': ['wa'],
     'id': 'bccbad82-abc4-4712-bd29-3e194e7a8042'},
    'imported': '2026-05-30T14:47:38.827030Z',
    'organization_type

In [127]:
def get_URLs_from_dist(arr):
    urls = []
    if arr:
        for a in arr:
            try: 
                urls.append(a['accessURL'])
            except:
                pass
            try: 
                urls.append(a['downloadURL'])
            except:
                pass
    
    urls = list(set(urls))
    if len(urls) > 0:
        url_out = ", ".join(urls)
    else:
        url_out = ""
    return url_out


data_gov_missing = pd.DataFrame(resp.json()['documents'])
metadata = pd.json_normalize(data_gov_missing['metadata'])
df_expanded = data_gov_missing.drop(columns=['metadata']).join(metadata)
df_expanded = df_expanded[['id', 'name', 'description',
'organization_type', 'organization_name', 'publisher', 
'last_seen', 'dcat.landingPage','distribution_count',
'dcat.distribution']]

df_expanded['dcat.distribution'] = df_expanded['dcat.distribution'].fillna("")
df_expanded['dcat.landingPage'] = df_expanded['dcat.landingPage'].fillna("")
df_expanded['download_urls'] = df_expanded['dcat.distribution'].apply(get_URLs_from_dist)

df_expanded['last_seen'] = pd.to_datetime(df_expanded['last_seen'], format='ISO8601')
df_expanded['last_seen'] = df_expanded['last_seen'].dt.date

df_expanded = df_expanded.drop(columns=['dcat.distribution','distribution_count']).rename(columns={'dcat.landingPage':'source_url','id':'hashed_id'})

In [ ]:
df_expanded.last_seen = df_expanded.last_seen.astype('str')

In [ ]:
for r in df_expanded_json:
    requests.post(
    "https://baserow.datarescueproject.org/api/database/rows/table/1054/?user_field_names=true",
    headers={
        "Authorization": f"Token {BASEROW_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    },
    json=r
    )

KeyboardInterrupt: 

In [7]:
def get_results_json(url, api_key=BASEROW_ACCESS_TOKEN):
    table = requests.get(
        url,
        headers={
            "Authorization": f"Token {api_key}"
        }
    )

    res = table.json()['results']
    if table.json()['next'] is not None:
        res.extend(get_results_json(table.json()['next']))

    return res

In [ ]:
rows_on_baserow = get_results_json("https://baserow.datarescueproject.org/api/database/rows/table/1054/?user_field_names=true")
rows_on_baserow = pd.DataFrame(rows_on_baserow)
rows_on_baserow = rows_on_baserow.drop(columns=['order'])
rows_to_add = df_expanded[~df_expanded.hashed_id.isin(rows_on_baserow.hashed_id)]
rows_to_add_json = rows_to_add.to_dict(orient='records')

In [ ]:
requests.post(
    "https://baserow.datarescueproject.org/api/database/rows/table/1054/?user_field_names=true",
    headers={
        "Authorization": f"Token {BASEROW_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    },
    json=rows_to_add_json[0]
    )

<Response [200]>

In [187]:
rows_to_toggle_status = rows_on_baserow[~rows_on_baserow.hashed_id.isin(df_expanded.hashed_id)]

In [197]:
len(rows_to_toggle_status)

0

In [ ]:
for i in rows_to_toggle_status.id:
    requests.patch(
        "https://baserow.datarescueproject.org/api/database/rows/table/1054/{i}/?user_field_names=true",
        headers={
            "Authorization": f"Token {BASEROW_ACCESS_TOKEN}",
            "Content-Type": "application/json"
        },
        json={
            "status":"Returned to data.gov"
            "last_seen": str(dt.date.today())
        }
    )

<Response [200]>

In [202]:
requests.get("https://data.iowa.gov/d/4y7u-nzj9")

<Response [404]>

In [ ]:
tracker_table_ids = [640, 901]
volunteer_table_ids = [1125, 1124]
Vol_API = ""
BASEROW_ACCESS_TOKEN = ""


In [185]:
def extract_link_values(value):
    if isinstance(value, dict):
        return value.get("value", value)

    if isinstance(value, list):
        return [
            item.get("value", item) if isinstance(item, dict) else item
            for item in value
        ]
        
    return value

def replace_linked_records_with_values(rows):
    """
    Convert:
        [{"id": 12, "value": "Alice"}]
    into:
        ["Alice"]
    """
    converted_rows = []
    fields = rows[0].keys()
    for row in rows:
        converted_row = row.copy()

        for field in fields:
            converted_row[field] = extract_link_values(converted_row.get(field))

        converted_rows.append(converted_row)

    return converted_rows

import numpy as np

def arrays_to_strings(df):
    converted = df.copy()

    def convert(value):
        if isinstance(value, (list, tuple, np.ndarray)):
            return ",".join(
                str(item.get("value", item))
                if isinstance(item, dict)
                else str(item)
                for item in value
            )
        return value

    for column in converted.columns:
        converted[column] = converted[column].map(convert)

    return converted

In [191]:
rows_on_tracker = get_results_json("https://baserow.datarescueproject.org/api/database/rows/table/640/?user_field_names=true",BASEROW_ACCESS_TOKEN)
rows_on_volunteer = get_results_json("https://baserow.datarescueproject.org/api/database/rows/table/1125/?user_field_names=true",Vol_API)

rows_on_tracker = replace_linked_records_with_values(rows_on_tracker)
rows_on_tracker = pd.DataFrame(rows_on_tracker)
rows_on_tracker = rows_on_tracker.drop(columns=['order'])
rows_on_tracker = arrays_to_strings(rows_on_tracker)
if rows_on_volunteer:
    rows_on_volunteer = replace_linked_records_with_values(rows_on_volunteer)
    rows_on_volunteer = pd.DataFrame(rows_on_volunteer)
    rows_on_volunteer = rows_on_volunteer.drop(columns=['order'])
    rows_on_volunteer = arrays_to_strings(rows_on_volunteer)
else:
    rows_on_volunteer = rows_on_tracker.iloc[0:0].copy()


In [ ]:
cols = list(set(rows_on_tracker.columns).intersection(rows_on_volunteer.columns))
cols = [c for c in cols if c!='id']
merged = rows_on_tracker.merge(rows_on_volunteer,on=cols,how="outer",indicator=True)
## if in both, don't have to do anything
# merged = merged[merged._merge!="both"]
## if in left, add to destination
to_add = merged[merged._merge=="left_only"].id_x.to_list()
to_add_rows = arrays_to_strings(rows_on_tracker[rows_on_tracker.id.isin(to_add)].drop(columns=['id'])).to_dict(orient="records")
response = requests.post(
    "https://baserow.datarescueproject.org/api/database/rows/table/1124/batch/?user_field_names=true",
    headers={
        "Authorization": f"Token {Vol_API}",
        "Content-Type": "application/json"
    },
    json={
        "items":to_add_rows
    }
)
print(response.json())
## if in right, delete from destination
to_delete = merged[merged._merge=="right_only"].id_y.to_list()

response = requests.post(
    "https://baserow.datarescueproject.org/api/database/rows/table/1124/batch-delete/",
    headers={
        "Authorization": f"Token {Vol_API}",
        "Content-Type": "application/json"
    },
    json={
        "items":to_delete
    }
)
print(response.json())


In [89]:
response = requests.post(
    "https://baserow.datarescueproject.org/api/database/rows/table/1124/?user_field_names=true",
    headers={
        "Authorization": f"Token {Vol_API}",
        "Content-Type": "application/json"
    },
    json=rows_to_add_json[0]
    )

In [70]:
import json
print("Status:", response.status_code)
print("Response:", json.dumps(response.json(), indent=2))
print("Payload:", json.dumps(payload, indent=2))

Status: 400
Response: {
  "error": "ERROR_REQUEST_BODY_VALIDATION",
  "detail": {
    "non_field_errors": [
      {
        "error": "Invalid data. Expected a dictionary, but got list.",
        "code": "invalid"
      }
    ]
  }
}


NameError: name 'payload' is not defined

In [24]:
r = requests.get(
    "https://baserow.datarescueproject.org/api/database/fields/table/1124/",
    headers={
        "Authorization": f"Token {Vol_API}",
    }
)

In [26]:
r.json()

[{'id': 10804,
  'table_id': 1124,
  'name': 'Name',
  'order': 0,
  'type': 'text',
  'primary': True,
  'read_only': False,
  'immutable_type': False,
  'immutable_properties': False,
  'description': None,
  'database_id': 320,
  'workspace_id': 182,
  'db_index': False,
  'field_constraints': [],
  'text_default': ''},
 {'id': 10805,
  'table_id': 1124,
  'name': 'Notes',
  'order': 1,
  'type': 'long_text',
  'primary': False,
  'read_only': False,
  'immutable_type': False,
  'immutable_properties': False,
  'description': None,
  'database_id': 320,
  'workspace_id': 182,
  'db_index': False,
  'field_constraints': [],
  'long_text_enable_rich_text': False},
 {'id': 10806,
  'table_id': 1124,
  'name': 'Active',
  'order': 2,
  'type': 'boolean',
  'primary': False,
  'read_only': False,
  'immutable_type': False,
  'immutable_properties': False,
  'description': None,
  'database_id': 320,
  'workspace_id': 182,
  'db_index': False,
  'field_constraints': [],
  'boolean_default